In [6]:
## This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/alx-exam-claims-data/claims_data.csv


In [59]:
from scipy.stats import norm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

**What proportion of individuals in this dataset would be classified as overweight or obese (BMI of greater than 25)?**

* 73%
* 82%
* 18%
* 85%
* 15%

In [8]:
# Load the dataset
df = pd.read_csv('/kaggle/input/alx-exam-claims-data/claims_data.csv', encoding='ISO-8859-1', index_col=0)
df.head()

,sex,bmi,steps,children,smoker,region,insurance_claim,claim_amount
age,,,,,,,,
19,female,27.900,3009,0,yes,southwest,yes,16884.9240
18,male,33.770,3008,1,no,southeast,yes,1725.5523
28,male,33.000,3009,3,no,southeast,no,0.0000
33,male,22.705,10009,0,no,northwest,no,0.0000
32,male,28.880,8010,0,no,northwest,yes,3866.8552


In [9]:
# Assuming the BMI column is named 'BMI'
overweight_or_obese = df[df['bmi'] > 25]
proportion = len(overweight_or_obese) / len(df) * 100
print(f"Proportion of individuals classified as overweight or obese: {proportion:.2f}%")

Proportion of individuals classified as overweight or obese: 81.54%


**Is the Poisson distribution a good choice to model the distribution of the number of children in this dataset?**

* No, the variance is significantly lower than the mean, suggesting underdispersion relative to the Poisson distribution.
* No, because the Poisson only applies to positive integers, so cannot accommodate observations with 0 children.
* No, the variance is significantly higher than the mean, suggesting overdispersion relative to the Poisson distribution.
* Yes, the Poisson is a good choice for count data such as the number of children in a family.
* No, the Poisson is inappropriate, as it is a continuous distribution while the number of children is a discrete variable.

In [10]:
mean_children = df['children'].mean()
mean_children

1.0949177877429

In [11]:
variance_children = df['children'].var()
variance_children

1.453212745666905

Use the appropriate model from the sklearn library (with default parameters unless specified otherwise) to fit a logistic regression model to the data, with insurance_claim as your target variable, using all other fields apart from claim_amount and creating dummy variables for the categorical variables in the data, dropping the first in each instance.

Do a test-train split holding out 33% of the data for the test set, using a random seed of 42 for the split.
Convert your target variable to a binary 0 or 1, where 1 indicates that there was a claim.
**What proportion of claim indicators in the test set are correctly predicted?**

* 86%
* 87%
* 81%
* 91%

In [12]:
# Convert categorical variables to dummy variables
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

In [13]:
# Convert target variable to binary
df['insurance_claim'] = df['insurance_claim'].map({'yes': 1, 'no': 0})

In [14]:
# Separate features and target variable
X = df.drop(columns=['insurance_claim', 'claim_amount'])
y = df['insurance_claim']

In [15]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)


In [16]:
# Fit a logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [17]:
# Predict on the test set
y_pred = model.predict(X_test)


In [18]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy_percentage = accuracy * 100

accuracy_percentage

86.19909502262443

Now fit another logistic regression, this time using the statsmodels library to do so, with default parameters.

Be sure to add a constant to your X matrices, both train and test (you might want to check the statsmodels documentation for the add_constant function).

**Which of the following best summarises the results?**

* Age, sex, BMI, number of children and smoker status significantly affect the likelihood of an insurance claim. Number of steps is not significant. There appear to be some regional effects, but these are not strongly significant.
* Age, BMI, number of children and smoker status significantly affect the likelihood of an insurance claim. Number of steps and sex are not significant. There appear to be some regional effects, but these are not strongly significant.
* All features seem to be significant predictors of the likelihood of a claim.
* Apart from age, BMI and smoker status, none of the other features are statistically significant predictors of an insurance claim.
* Age, sex, BMI and smoker status significantly affect the likelihood of an insurance claim. Number of steps and number of children are not significant. There appear to be some regional effects, but these are not strongly significant.

In [19]:
!pip install pandas numpy statsmodels

In [26]:
 X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               1000 non-null   int64  
 1   bmi               1000 non-null   float64
 2   children          1000 non-null   int64  
 3   steps             1000 non-null   int64  
 4   sex_male          1000 non-null   bool   
 5   smoker_yes        1000 non-null   bool   
 6   region_northwest  1000 non-null   bool   
 7   region_southeast  1000 non-null   bool   
 8   region_southwest  1000 non-null   bool   
dtypes: bool(5), float64(1), int64(3)
memory usage: 36.3 KB


In [34]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
age                 0
bmi                 0
children            0
steps               0
claim               0
sex_male            0
smoker_yes          0
region_northwest    0
region_southeast    0
region_southwest    0
dtype: int64


In [35]:
# Ensure all boolean data is converted to int
df = df.astype({'sex_male': int, 'smoker_yes': int, 'region_northwest': int, 'region_southeast': int, 'region_southwest': int})


In [36]:
# Add a constant to the feature matrix
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

In [37]:
# Fit a logistic regression model using statsmodels
logit_model = sm.Logit(y_train.astype(float), X_train_sm.astype(float))
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.686896
         Iterations 4


In [39]:
# Print the summary
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                  claim   No. Observations:                  800
Model:                          Logit   Df Residuals:                      790
Method:                           MLE   Df Model:                            9
Date:                Sun, 04 Aug 2024   Pseudo R-squ.:                0.008478
Time:                        20:45:30   Log-Likelihood:                -549.52
converged:                       True   LL-Null:                       -554.22
Covariance Type:            nonrobust   LLR p-value:                    0.4014
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.6422      0.464      1.384      0.166      -0.267       1.552
age                 -0.0018      0.005     -0.334      0.738      -0.012       0.009
bmi                 -0.0145 

Now fit a random forest with 100 trees and a random seed of 101, and default parameters for the rest.

**Which of the following sets out the number of false negatives and false positives in the confusion matrix on the test data?**

* FN = 8, FP = 5
* FN = 5, FP = 8
* FN = 245, FP = 177
* FN = 177, FP = 245
* FN = 12, FP = 12

In [54]:
# Fit a random forest model with 100 trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=101)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=101)

In [55]:
# Predict on the test set
y_pred = rf_model.predict(X_test)

In [56]:
# Compute the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = conf_matrix.ravel()

# Print false negatives and false positives
print(f"False Negatives (FN): {fn}")
print(f"False Positives (FP): {fp}")

False Negatives (FN): 43
False Positives (FP): 45


Fit Support Vector Machine models to the training data, using respectively the radial, sigmoid and linear kernels with default parameters.

**Which model yields the best accuracy on test data?**

* Sigmoid and linear yield very similar accuracies
* Radial and linear yield very similar accuracies
* Linear
* Radial
* Sigmoid

In [60]:
# Fit SVM models with different kernels
kernels = ['linear', 'rbf', 'sigmoid']
accuracies = {}

for kernel in kernels:
    svm_model = SVC(kernel=kernel, random_state=101)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies[kernel] = accuracy
    print(f"Accuracy with {kernel} kernel: {accuracy:.4f}")

# Determine which kernel yields the best accuracy
best_kernel = max(accuracies, key=accuracies.get)
print(f"The best kernel is: {best_kernel} with an accuracy of {accuracies[best_kernel]:.4f}")

Accuracy with linear kernel: 0.5050
Accuracy with rbf kernel: 0.5300
Accuracy with sigmoid kernel: 0.4600
The best kernel is: rbf with an accuracy of 0.5300
